# Topic gap finder

Compare your page with competing pages and list the terms they share that your page never uses.

1. Replace the sample text in the next cell with your page and at least two competitors.
2. Choose **Runtime > Run all**.
3. Read the list and decide which terms belong on your page. A gap is a prompt to check, not an instruction to add the word.

This compares words and two-word phrases. It doesn't recognise entities or meaning, so synonyms count as different terms.

To keep your changes, choose **File > Save a copy in Drive**.

In [ ]:
MY_PAGE = """
Semantic SEO helps search engines understand what a page is about.
Write about one topic in depth and link related pages together.
"""

COMPETITORS = [
    """
    Semantic SEO starts with entities. Search engines connect each entity to its attributes
    in a knowledge graph. Structured data helps state those facts clearly, and a topical map
    shows which pages a site still needs.
    """,
    """
    A topical map groups pages by topic so internal links follow real relationships.
    Cover the attributes people ask about for each entity, and add structured data
    where it matches the visible content.
    """,
    """
    Good semantic content names the main entity early and answers related questions.
    Search engines use structured data and internal links to work out how pages connect.
    """,
]

MIN_COMPETITORS = 2  # a term must appear on at least this many competing pages
MAX_RESULTS = 30

In [ ]:
import re
from collections import Counter

STOPWORDS = set("""
a about all also an and are as at be because been but by can do does each for from had has have how
if in into is it its more most much not of on one or our out so some such than that the their them
then there these they this those to too up us use used using was we were what when where which who
why will with would you your
""".split())


def terms(text):
    """Count single words and two-word phrases, skipping common words."""
    found = Counter()
    for sentence in re.split(r"[.!?;:\n]+", text.lower()):
        words = re.findall(r"[a-z0-9][a-z0-9'-]*", sentence)
        found.update(w for w in words if w not in STOPWORDS and len(w) > 2)
        found.update(f"{a} {b}" for a, b in zip(words, words[1:]) if a not in STOPWORDS and b not in STOPWORDS)
    return found


def topic_gaps(my_page, competitors, min_competitors=2):
    """Return (term, pages using it, total mentions) for terms missing from my_page."""
    mine = terms(my_page)
    pages, mentions = Counter(), Counter()
    for page in competitors:
        counts = terms(page)
        pages.update(counts.keys())
        mentions.update(counts)
    gaps = [(t, n, mentions[t]) for t, n in pages.items() if n >= min_competitors and t not in mine]
    return sorted(gaps, key=lambda g: (-g[1], -g[2], g[0]))


assert topic_gaps("cats", ["cats and knowledge graph", "knowledge graph"]) == [
    ("graph", 2, 2), ("knowledge", 2, 2), ("knowledge graph", 2, 2)
]

In [ ]:
gaps = topic_gaps(MY_PAGE, COMPETITORS, MIN_COMPETITORS)

if not gaps:
    print("No gaps found. Try lowering MIN_COMPETITORS or adding more competing pages.")
else:
    print(f"{'Term':<32}{'Pages':>6}{'Mentions':>10}")
    for term, page_count, total in gaps[:MAX_RESULTS]:
        print(f"{term:<32}{page_count:>6}{total:>10}")